In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-32B-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.8: Fast Qwen3 patching. Transformers: 4.57.3.
   \\   /|    Tesla V100-SXM2-32GB. Num GPUs = 1. Max memory: 31.739 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, 
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing= "unsloth",
    random_state= 3407,
    use_rslora= False,
    loftq_config= None    
)

Unsloth 2025.12.8 patched 64 layers with 64 QKV layers, 64 O layers and 64 MLP layers.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("kyujinpy/KoCoT_2000", split="train")

In [ ]:
dataset

In [ ]:
dataset[0]

{'source': "이 과제에서는 입력 목록 A가 주어집니다. 목록에서 숫자의 개수가 알파벳의 개수보다 많으면 '숫자가 이긴다'라고 답하십시오. 알파벳의 개수가 목록의 숫자 개수보다 많으면 '알파벳이 이긴다'라고 답합니다. 숫자의 개수와 목록의 알파벳의 개수가 같으면 '숫자와 알파벳 동점'이라고 답하세요.\n\n['9047', 'C', 't', '5181', '5871', 'n', '7319', '283', '4997', 'n', '4577', 'K', 'B', 'Y', 'R', '8411', 'D', 'G', 'A', '5667', 'e', 'i', '805', '763', 'm', '9673', '1043', 'N', 'r']",
 'target': 'Alphabets Win',
 'rationale': "목록 ['9047', 'C', 't', '5181', '5871', 'n', '7319', '283', '4997', 'n', '4577', 'K', 'B', 'Y', 'R', '8411']이 주어지면 숫자 값 6개와 문자 14개가 존재합니다. 알파벳의 개수가 숫자의 개수보다 많으므로 답은 알파벳이 승리해야 합니다.",
 'task': 'synthetic',
 'type': 'CoT'}

In [ ]:
import json

def generation_conversation(examples):
    problems = examples["source"]
    solutions = examples["rationale"]
    answers = examples["target"]
    
    conversations = []
    for problem, solution, answer in zip(problems, solutions, answers):
        print(f"problem: {problem}")
        print(f"solution: {solution}")
        print(f"answer: {answer}")
        assistant_output = {
            "reasoning": solution.strip(),
            "answer": answer
        }
        conversations.append([
            {
                "role" : "user", 
                "content" : problem
            },
            {
                "role" : "assistant", 
                "content": json.dumps(
                    assistant_output,
                    ensure_ascii=False
                )
            }
        ])
    return {"conversation": conversations}
    
dataset = dataset.map(generation_conversation, batched = True)

Map:   0%|          | 0/2159 [00:00<?, ? examples/s]

problem: 이 과제에서는 입력 목록 A가 주어집니다. 목록에서 숫자의 개수가 알파벳의 개수보다 많으면 '숫자가 이긴다'라고 답하십시오. 알파벳의 개수가 목록의 숫자 개수보다 많으면 '알파벳이 이긴다'라고 답합니다. 숫자의 개수와 목록의 알파벳의 개수가 같으면 '숫자와 알파벳 동점'이라고 답하세요.

['9047', 'C', 't', '5181', '5871', 'n', '7319', '283', '4997', 'n', '4577', 'K', 'B', 'Y', 'R', '8411', 'D', 'G', 'A', '5667', 'e', 'i', '805', '763', 'm', '9673', '1043', 'N', 'r']
solution: 목록 ['9047', 'C', 't', '5181', '5871', 'n', '7319', '283', '4997', 'n', '4577', 'K', 'B', 'Y', 'R', '8411']이 주어지면 숫자 값 6개와 문자 14개가 존재합니다. 알파벳의 개수가 숫자의 개수보다 많으므로 답은 알파벳이 승리해야 합니다.
answer: Alphabets Win
problem: 이 과제에서는 주어진 문자로 시작하는 문장의 단어 수를 세어야 합니다. 단어가 아닌 숫자로 답하세요.

문장: '큰 도자기 장식품이 화단과 여러 꽃 덤불과 관목으로 둘러싸인 밝은 흐린 날에'. 문장에서 문자 'f'로 시작하는 단어가 몇 개나 되는지 맞히세요.
solution: '화창한 흐린 날에 커다란 도자기 장식이 화단과 여러 꽃 덤불과 관목으로 둘러싸여 있다'라는 문장이 주어졌을 때, 한 단어씩 시도해 봅시다.\n1. 'a' : 아니요 -> (총) 0\n2. 'large' : 아니요 -> (총) 0\n3. 'ceramic' : No -> (총) 0\n4. 'ornament' : No -> (총) 0\n5. 'is': No ->(total)0 \n6.'둘러싸인':No->(total)0 \n7.'by':No->(total)\b 8.'a':No->(total

In [ ]:
dataset[0]

{'source': "이 과제에서는 입력 목록 A가 주어집니다. 목록에서 숫자의 개수가 알파벳의 개수보다 많으면 '숫자가 이긴다'라고 답하십시오. 알파벳의 개수가 목록의 숫자 개수보다 많으면 '알파벳이 이긴다'라고 답합니다. 숫자의 개수와 목록의 알파벳의 개수가 같으면 '숫자와 알파벳 동점'이라고 답하세요.\n\n['9047', 'C', 't', '5181', '5871', 'n', '7319', '283', '4997', 'n', '4577', 'K', 'B', 'Y', 'R', '8411', 'D', 'G', 'A', '5667', 'e', 'i', '805', '763', 'm', '9673', '1043', 'N', 'r']",
 'target': 'Alphabets Win',
 'rationale': "목록 ['9047', 'C', 't', '5181', '5871', 'n', '7319', '283', '4997', 'n', '4577', 'K', 'B', 'Y', 'R', '8411']이 주어지면 숫자 값 6개와 문자 14개가 존재합니다. 알파벳의 개수가 숫자의 개수보다 많으므로 답은 알파벳이 승리해야 합니다.",
 'task': 'synthetic',
 'type': 'CoT',
 'conversation': [{'content': "이 과제에서는 입력 목록 A가 주어집니다. 목록에서 숫자의 개수가 알파벳의 개수보다 많으면 '숫자가 이긴다'라고 답하십시오. 알파벳의 개수가 목록의 숫자 개수보다 많으면 '알파벳이 이긴다'라고 답합니다. 숫자의 개수와 목록의 알파벳의 개수가 같으면 '숫자와 알파벳 동점'이라고 답하세요.\n\n['9047', 'C', 't', '5181', '5871', 'n', '7319', '283', '4997', 'n', '4577', 'K', 'B', 'Y', 'R', '8411', 'D', 'G', 'A', '5667', 'e', 'i', '805', '763', 'm', '9673', '1043', 'N', 'r']",


In [ ]:
def formatting_prompts_func(examples):
    convs = examples["conversation"]
    texts = [tokenizer.apply_chat_template(conv, tokenize=False, enable_thinking = False) for conv in convs]
    return {"text" : texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/2159 [00:00<?, ? examples/s]

In [ ]:
dataset[0]

{'source': "이 과제에서는 입력 목록 A가 주어집니다. 목록에서 숫자의 개수가 알파벳의 개수보다 많으면 '숫자가 이긴다'라고 답하십시오. 알파벳의 개수가 목록의 숫자 개수보다 많으면 '알파벳이 이긴다'라고 답합니다. 숫자의 개수와 목록의 알파벳의 개수가 같으면 '숫자와 알파벳 동점'이라고 답하세요.\n\n['9047', 'C', 't', '5181', '5871', 'n', '7319', '283', '4997', 'n', '4577', 'K', 'B', 'Y', 'R', '8411', 'D', 'G', 'A', '5667', 'e', 'i', '805', '763', 'm', '9673', '1043', 'N', 'r']",
 'target': 'Alphabets Win',
 'rationale': "목록 ['9047', 'C', 't', '5181', '5871', 'n', '7319', '283', '4997', 'n', '4577', 'K', 'B', 'Y', 'R', '8411']이 주어지면 숫자 값 6개와 문자 14개가 존재합니다. 알파벳의 개수가 숫자의 개수보다 많으므로 답은 알파벳이 승리해야 합니다.",
 'task': 'synthetic',
 'type': 'CoT',
 'conversation': [{'content': "이 과제에서는 입력 목록 A가 주어집니다. 목록에서 숫자의 개수가 알파벳의 개수보다 많으면 '숫자가 이긴다'라고 답하십시오. 알파벳의 개수가 목록의 숫자 개수보다 많으면 '알파벳이 이긴다'라고 답합니다. 숫자의 개수와 목록의 알파벳의 개수가 같으면 '숫자와 알파벳 동점'이라고 답하세요.\n\n['9047', 'C', 't', '5181', '5871', 'n', '7319', '283', '4997', 'n', '4577', 'K', 'B', 'Y', 'R', '8411', 'D', 'G', 'A', '5667', 'e', 'i', '805', '763', 'm', '9673', '1043', 'N', 'r']",


In [ ]:
dataset[0]["text"]

In [ ]:
import wandb

wandb.init(
    project="SFT_CoT",
    name = "test"
)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset= dataset,
    eval_dataset= None,
    args = SFTConfig(
        dataset_text_field= "text",
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps= 10,
        optim = "adamw_8bit",
        weight_decay= 0.001,
        lr_scheduler_type="linear",
        seed = 3407,
        report_to="wandb"
    )
)

[trl.trainer.sft_trainer|WARNING]You are using a per_device_train_batch_size of 1 with padding-free training. Using a batch size of 1 anihilate the benefits of padding-free training. Please consider increasing the batch size to at least 2.


Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/2159 [00:00<?, ? examples/s]

[accelerate.utils.other|WARNING]Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n<think>\n\n"
)

Map (num_proc=12):   0%|          | 0/2159 [00:00<?, ? examples/s]

In [ ]:
sum(trainer.train_dataset[0]["attention_mask"])


In [ ]:
len(trainer.train_dataset[0]['attention_mask'])

In [ ]:
trainer.train_dataset[0]["input_ids"]

[151644,
 872,
 198,
 12802,
 45130,
 120,
 37087,
 129889,
 42349,
 134952,
 49664,
 362,
 19969,
 55673,
 31079,
 126886,
 21953,
 13,
 134952,
 49664,
 56475,
 69192,
 92187,
 20401,
 73523,
 135444,
 125214,
 126793,
 144923,
 20401,
 73523,
 23259,
 129885,
 126735,
 89940,
 364,
 123967,
 92187,
 19969,
 23084,
 133507,
 13146,
 6,
 129254,
 143603,
 16186,
 139713,
 13,
 125214,
 126793,
 144923,
 20401,
 73523,
 135444,
 134952,
 49664,
 20401,
 69192,
 92187,
 73523,
 23259,
 129885,
 126735,
 89940,
 364,
 144135,
 126793,
 144923,
 12802,
 23084,
 133507,
 13146,
 6,
 129254,
 143603,
 60838,
 13,
 69192,
 92187,
 20401,
 73523,
 23259,
 80573,
 134952,
 49664,
 20401,
 125214,
 126793,
 144923,
 20401,
 73523,
 135444,
 78374,
 89940,
 364,
 123967,
 92187,
 80573,
 125214,
 126793,
 144923,
 126322,
 126333,
 6,
 130939,
 143603,
 91145,
 382,
 677,
 24,
 15,
 19,
 22,
 516,
 364,
 34,
 516,
 364,
 83,
 516,
 364,
 20,
 16,
 23,
 16,
 516,
 364,
 20,
 23,
 22,
 16,
 516,
 

In [13]:
trainer.train_dataset[0]['labels']

[-100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,

In [ ]:
lengths =  [
    len(tokenizer(x["text"])["input_ids"])
    for x in dataset
]

print(sum(lengths)/len(lengths))
print(len(lengths))
print(max(lengths))
print(min(lengths))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(8, 5))
plt.hist(lengths, bins=50)
plt.xlabel("Number of tokens")
plt.ylabel("Number of samples")
plt.title("Token length distribution")
plt.show()


In [ ]:
tokenizer.decode(trainer.train_dataset[0]['input_ids'])

In [27]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")

In [28]:
trainer_stats = trainer.train()

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-32B-bnb-4bit",
    max_seq_length = 1024,
    load_in_4bit = True,
)

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")